# Generate flanking primers for genes in ecoli


In [ ]:
! pip install biopython

In [27]:
import pandas as pd
import sys
sys.path.append('/Users/amora/Documents/code/oligopoolio')
from oligopoolio.oligopools import *
from oligopoolio.primers import *
from oligopoolio.oligos import *
import pyswarms as ps
import numpy as np
import itertools
from primer3 import calc_hairpin, calc_homodimer


In [65]:
test_data_dir = '../tests/data/'
gff_file = f'{test_data_dir}genome_NEB_B/genomic.gff'
reference_fasta = f'{test_data_dir}genome_NEB_B/GCF_001559615.2_ASM155961v2_genomic.fna'
gene_name = 'yigF'

def gc_content(seq):
    return (seq.count('G') + seq.count('C')) / len(seq)

def melting_temp(seq):
    # Basic Wallace rule
    return 2 * (seq.count('A') + seq.count('T')) + 4 * (seq.count('G') + seq.count('C'))
    
def reverse_complement(seq):
    """Generate the reverse complement of a DNA sequence."""
    seq = seq.strip().replace(' ', '') # Do a little cleaning
    complement = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
    return ''.join([complement[base] for base in reversed(seq)])

overhang = 15
seqid, start, end, strand, upstream_flank, downstream_flank, gene_seq = get_flanking_primers(gene_name,
                                                                                             gff_file,
                                                                                             reference_fasta, 
                                                                                             overhang, 
                                                                                             overhang) # The upstream and downstream flanks

assert gene_seq[:3] == 'ATG'  # i.e. methionine
assert gene_seq[-3:] in ['TAA', 'TGA']
# Now we want to reverse complement the downstream flank + overhang with the gene
fwd_primer = upstream_flank + gene_seq[:overhang]
rev_primer = reverse_complement(gene_seq[-overhang:] + downstream_flank)

for seq in fwd_primer, rev_primer:
    gc = gc_content(seq)
    tm = melting_temp(seq)
    
    results = check_secondary_structure(seq)
    homodimer_tm = results['homodimer']['homodimer_dg']
    hairpin_tm = results['hairpin']['hairpin_dg']
    
    # Penalize if Tm deviates from 60
    primer_tm = primer3.bindings.calcTm(seq)
    tm_penalty = (primer_tm - 60)**2
    
    # Penalize if GC content is far from 0.5
    gc_penalty = (gc - 0.5)**2
    
    # Example: simple penalty for homopolymer runs
    max_homopolymer = max(len(list(g)) for _, g in itertools.groupby(seq))
    homo_penalty = 10 * max(0, max_homopolymer - 4)
    print('GC', gc, 'Melting', tm, 'primer_tm', primer_tm, 'homodimer_tm', homodimer_tm, 'hairpin_tm', hairpin_tm)

GC 0.26666666666666666 Melting 76 primer_tm 56.66836489870229 homodimer_tm 0.46655683520280794 hairpin_tm 0.0
GC 0.4666666666666667 Melting 88 primer_tm 67.63692942359881 homodimer_tm -2.8172531648004773 hairpin_tm 0.5834328352028048


In [63]:
rev_primer

'TAAGCCACCTGTTATTCAAAGGCTCCAGGT'

In [62]:
fwd_primer

'AGTTATGGAGTATTTATGAGTAAGGAATAT'

In [67]:
test_data_dir = '../tests/data/'
gff_file = f'{test_data_dir}genome_NEB_B/genomic.gff'
reference_fasta = f'{test_data_dir}genome_NEB_B/GCF_001559615.2_ASM155961v2_genomic.fna'
gene_name = 'frsA'

overhang = 15
seqid, start, end, strand, upstream_flank, downstream_flank, gene_seq = get_flanking_primers(gene_name,
                                                                                             gff_file,
                                                                                             reference_fasta, 
                                                                                             overhang, 
                                                                                             overhang) # The upstream and downstream flanks

assert gene_seq[:3] == 'ATG'  # i.e. methionine
assert gene_seq[-3:] in ['TAA', 'TGA']
# Now we want to reverse complement the downstream flank + overhang with the gene
fwd_primer = upstream_flank + gene_seq[:overhang]
rev_primer = reverse_complement(gene_seq[-overhang:] + downstream_flank)

for seq in fwd_primer, rev_primer:
    gc = gc_content(seq)
    tm = melting_temp(seq)
    
    results = check_secondary_structure(seq)
    homodimer_tm = results['homodimer']['homodimer_dg']
    hairpin_tm = results['hairpin']['hairpin_dg']
    
    # Penalize if Tm deviates from 60
    primer_tm = primer3.bindings.calcTm(seq)
    tm_penalty = (primer_tm - 60)**2
    
    # Penalize if GC content is far from 0.5
    gc_penalty = (gc - 0.5)**2
    
    # Example: simple penalty for homopolymer runs
    max_homopolymer = max(len(list(g)) for _, g in itertools.groupby(seq))
    homo_penalty = 10 * max(0, max_homopolymer - 4)
    print(seq, 'GC', gc, 'Melting', tm, 'primer_tm', primer_tm, 'homodimer_tm', homodimer_tm, 'hairpin_tm', hairpin_tm)

CTGGAGGCTGCATCCATGACACAGGCAAAC GC 0.5666666666666667 Melting 94 primer_tm 71.0345072969223 homodimer_tm -5.990053270923067 hairpin_tm 0.42348533519951975
AAATTTAGCAAATTTTTAACACAAACGTTT GC 0.2 Melting 72 primer_tm 58.27997278193152 homodimer_tm -2.7624457178601385 hairpin_tm 0.867338446937077


In [68]:
test_data_dir = '../tests/data/'
gff_file = f'{test_data_dir}genome_NEB_B/genomic.gff'
reference_fasta = f'{test_data_dir}genome_NEB_B/GCF_001559615.2_ASM155961v2_genomic.fna'
gene_name = 'acpH'

overhang = 15
seqid, start, end, strand, upstream_flank, downstream_flank, gene_seq = get_flanking_primers(gene_name,
                                                                                             gff_file,
                                                                                             reference_fasta, 
                                                                                             overhang, 
                                                                                             overhang) # The upstream and downstream flanks

assert gene_seq[:3] == 'ATG'  # i.e. methionine
assert gene_seq[-3:] in ['TAA', 'TGA']
# Now we want to reverse complement the downstream flank + overhang with the gene
fwd_primer = upstream_flank + gene_seq[:overhang]
rev_primer = reverse_complement(gene_seq[-overhang:] + downstream_flank)

for seq in fwd_primer, rev_primer:
    gc = gc_content(seq)
    tm = melting_temp(seq)
    
    results = check_secondary_structure(seq)
    homodimer_tm = results['homodimer']['homodimer_dg']
    hairpin_tm = results['hairpin']['hairpin_dg']
    
    # Penalize if Tm deviates from 60
    primer_tm = primer3.bindings.calcTm(seq)
    tm_penalty = (primer_tm - 60)**2
    
    # Penalize if GC content is far from 0.5
    gc_penalty = (gc - 0.5)**2
    
    # Example: simple penalty for homopolymer runs
    max_homopolymer = max(len(list(g)) for _, g in itertools.groupby(seq))
    homo_penalty = 10 * max(0, max_homopolymer - 4)
    print(seq, 'GC', gc, 'Melting', tm, 'primer_tm', primer_tm, 'homodimer_tm', homodimer_tm, 'hairpin_tm', hairpin_tm)

GATAGAATATAATCGATGAATTTTTTAGCT GC 0.23333333333333334 Melting 74 primer_tm 55.644972396230685 homodimer_tm -2.475236941328655 hairpin_tm 0.0
GGATGAACTAACGTTTTATAACGCCTTGCG GC 0.43333333333333335 Melting 86 primer_tm 65.1113029158875 homodimer_tm -1.490839494398158 hairpin_tm -0.1315911648004761


In [49]:
search_term = 'esterase'
with open(f'{test_data_dir}genome_NEB_B/genomic_{search_term}.gff', 'w') as fout:
    with open(f'{test_data_dir}genome_NEB_B/genomic.gff', 'r+') as fin:
        for line in fin:
            if 'esterase' in line:
                fout.write(line)

In [39]:
! pip install bcbio-gff

In [44]:
filter_gff(f'{test_data_dir}genome_NEB_B/genomic.gff', 'esterase', 'CDS')

[]

In [29]:
f'{test_data_dir}genome_NEB_B/genomic.gff'

'../tests/data/genome_NEB_B/genomic.gff'

In [25]:
homodimer_tm

3.2204279525323387